
# OpenCV Vehicle Counting Baseline
This notebook provides an educational baseline for vehicle detection and counting using classical OpenCV methods (Background Subtraction, Contours).

**Limitations:**
- Sensitive to camera movement (unsuitable for most moving drone footage).
- Sensitive to shadows, perspective changes, and occlusion.
- Cannot easily classify vehicle types (car vs truck) compared to trained CNNs.


In [ ]:

import cv2
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))
from src.aeronetra.counting.drawing import draw_roi


In [ ]:

def process_video(video_path: str, output_path: str):
    if not Path(video_path).exists():
        print(f"Video {video_path} not found. Skipping execution.")
        return
        
    cap = cv2.VideoCapture(video_path)
    bg_subtractor = cv2.createBackgroundSubtractorMOG2(history=500, varThreshold=50, detectShadows=True)
    
    # ROI: (xmin, ymin, xmax, ymax)
    roi = (100, 100, 500, 400)
    
    frame_count = 0
    total_vehicles = 0
    
    while cap.isOpened() and frame_count < 100: # process 100 frames for demo
        ret, frame = cap.read()
        if not ret:
            break
        
        # Apply ROI manually for simplicity
        fg_mask = bg_subtractor.apply(frame)
        
        # Morphology
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
        fg_mask = cv2.morphologyEx(fg_mask, cv2.MORPH_OPEN, kernel)
        fg_mask = cv2.morphologyEx(fg_mask, cv2.MORPH_CLOSE, kernel)
        
        # Contours
        contours, _ = cv2.findContours(fg_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        for cnt in contours:
            area = cv2.contourArea(cnt)
            if area > 500: # Min area
                x, y, w, h = cv2.boundingRect(cnt)
                
                # Check ROI intersection
                cx = x + w/2
                cy = y + h/2
                if roi[0] < cx < roi[2] and roi[1] < cy < roi[3]:
                    cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)
                    total_vehicles += 1 # Note: this is frame detection, not tracking!
                    
        # Draw ROI
        frame = draw_roi(frame, roi)
        frame_count += 1
        
    cap.release()
    print(f"Processed {frame_count} frames. Detections inside ROI: {total_vehicles}")
    print("Remember: Frame-by-frame detections != unique traffic counts.")

# Example usage (commented out as we don't have video right now)
# process_video("../data/raw/sample.mp4", "../outputs/predictions/sample_out.mp4")
